# TinyML — a rede neural do motor, em TensorFlow Lite, **dentro** do ESP32

Mesmo dataset da coleta, mesma leitura do InfluxDB, mesmo corte por rodada e o mesmo scaler
da floresta. O que muda é o modelo: uma rede neural com o mesmo desenho da rede da API
(8 entradas, 16 neurônios, 4 saídas), treinada no TensorFlow, que é quem sabe gerar o
`.tflite`.

| Arquivo gerado | Vai para | Serve para |
|---|---|---|
| `modelo_motor.tflite` | seu computador | a rede, no formato do TensorFlow Lite |
| `ModeloMotorScaler.hpp` | o `src/` do firmware | a média e o desvio de cada feature |
| `ModeloMotorRede.hpp` | o `src/` do firmware | o mesmo `.tflite`, como um vetor de bytes em C++ |

Os dois `.hpp` saem do mesmo treino e viajam sempre juntos.

> **Execute a primeira célula de código duas vezes.** Na primeira, o Colab reinicia a sessão (e avisa que ela travou) para carregar as versões fixadas das bibliotecas; na segunda, ela passa direto.

In [ ]:
# EXECUTE ESTA CELULA DUAS VEZES: na 1a o Colab reinicia a sessao.
#
# Versoes fixas: as mesmas do notebook da floresta, mais o TensorFlow. O
# .tflite gerado aqui roda no interpretador da placa (MicroTFLite); a versao
# fixa garante que o arquivo saia igual no proximo semestre.
#
# influxdb3-python le por SQL (Arrow Flight); pyarrow vem junto e e quem
# converte o resultado em DataFrame.
!pip install -q "tensorflow==2.21.0" "influxdb3-python==0.21.0" "pyarrow==26.0.1" "matplotlib==3.11.2" "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1"

# Reinicia a sessao se o numpy da memoria nao for o recem-instalado
import numpy, os
from importlib.metadata import version
if numpy.__version__ != version("numpy"):
    print("NumPy trocado: reiniciando a sessao. Execute esta celula de novo.")
    os.kill(os.getpid(), 9)

## 1) Conectar no InfluxDB Cloud

Os mesmos valores do notebook da coleta — é o mesmo dataset, lido do mesmo lugar.
O `bucket` do Node-RED é o `database` aqui. Sem token, o notebook avisa e segue com o
dataset de referência (seção 2).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from influxdb_client_3 import InfluxDBClient3

INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"
INFLUX_BUCKET = "IoTSensores"          # o "database" no InfluxDB 3
MEASUREMENT   = "vibracao_multiclasse"
DEVICE        = ""                     # o MQTT_CLIENT_ID da sua placa; vazio = todas as placas do bucket

# A ORDEM desta lista e o contrato com o firmware: a entrada 0 da rede e
# mean_ax porque mean_ax e a primeira coluna aqui.
FEATURES = ["mean_ax", "mean_ay", "mean_az",
            "std_ax", "std_ay", "std_az",
            "std_mag", "p2p_mag"]

CLASSES = ["operando", "inclinado_frente", "inclinado_tras", "anomalia"]

## 2) Ler as janelas

O mesmo `SELECT` da coleta: as oito features, o rótulo e a rodada.

In [ ]:
colunas = ", ".join(f'"{f}"' for f in FEATURES)
filtro_device = f"AND \"device\" = '{DEVICE}'" if DEVICE else ""
sql = f'''
SELECT time, "device", "label", "rodada", {colunas}
FROM "{MEASUREMENT}"
WHERE time >= now() - INTERVAL '30 days'
{filtro_device}
ORDER BY time
'''

try:
    client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)
    df = client.query(query=sql, language="sql").to_pandas()
    client.close()
    df = (df.query("label in @CLASSES")
            .dropna(subset=FEATURES)
            .sort_values("time")
            .reset_index(drop=True))
    df["rodada"] = df["rodada"].astype(int)   # veio como tag, entao veio texto
    print(f"{len(df)} janelas")
    print(df.groupby(["device", "label", "rodada"]).size().to_string())
except Exception as erro:
    print("Nao consegui ler o InfluxDB:", erro)
    print("Siga com o Plano B, logo abaixo.")
    df = pd.DataFrame()

### Plano B: o dataset de referência do professor
Sem dataset próprio (ou com rodadas incompletas), use o dataset de referência: 4 classes × 3
rodadas × 30 janelas, coletado pelo professor com a mesma placa de referência. Mude
`USAR_REFERENCIA` para `True` e siga o notebook do mesmo jeito.

> Vale para a aula. **Não vale para o CP:** lá o dataset tem que ser o seu.

In [ ]:
USAR_REFERENCIA = False   # True = treinar com o dataset de referência em vez do seu

if USAR_REFERENCIA or len(df) == 0:
    URL_REFERENCIA = "https://raw.githubusercontent.com/norisjunior/FIAP-IoT/fiapiot/eval/app17-IMU-GerarDataset-Sinais/app17-7-MultiClassAccFeaturesInflux/colab/dataset_mpu_referencia.csv"
    df = pd.read_csv(URL_REFERENCIA)
    df["rodada"] = df["rodada"].astype(int)
    print("usando o dataset de referência:", len(df), "janelas")
    display(df.groupby(["label", "rodada"]).size().unstack())

## 3) Split por rodada — a última rodada fica de fora do treino

Idêntico ao do notebook da floresta.

In [ ]:
rodada_teste = df["rodada"].max()

treino = df[df["rodada"] != rodada_teste]
teste  = df[df["rodada"] == rodada_teste]
print(f"treino: {len(treino)} janelas | teste: rodada {rodada_teste}, {len(teste)} janelas")

## 4) Padronizar e numerar as classes

- **O mesmo `StandardScaler` da floresta**: `(valor − média) / desvio`, com a média e o
  desvio do treino. Na rede neural ele não é enfeite: sem ele, as entradas grandes dominam
  as pequenas. Na placa, o `Scaler::standardize()` faz a mesma conta antes da rede.
- **As classes viram números em ordem alfabética**: 0 = anomalia, 1 = inclinado_frente,
  2 = inclinado_tras, 3 = operando. A rede devolve 4 probabilidades nessa ordem, e quem dá
  nome a elas é o vetor `NOMES_CLASSES` do firmware — o mesmo da floresta.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(treino[FEATURES])
X_treino = scaler.transform(treino[FEATURES]).astype("float32")
X_teste  = scaler.transform(teste[FEATURES]).astype("float32")

ORDEM = sorted(df["label"].unique())                 # a ordem das saídas da rede
numero = {classe: i for i, classe in enumerate(ORDEM)}
y_treino = treino["label"].map(numero).to_numpy()
y_teste  = teste["label"].map(numero).to_numpy()

print("ordem das saídas da rede:", ORDEM)

## 5) Treinar a rede neural

O mesmo desenho da rede da API: 8 entradas → 16 neurônios (ReLU) → 4 saídas (softmax, uma
probabilidade por classe). São **212 pesos**: 8 × 16 + 16 na primeira camada, 16 × 4 + 4 na
segunda.

A diferença é a biblioteca. A rede da API foi treinada no scikit-learn, que não gera
`.tflite`. Esta é treinada no TensorFlow (Keras), que gera. A semente fixa faz o treino sair
igual a cada execução.

In [ ]:
import tensorflow as tf
import keras

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

rede = keras.Sequential([
    keras.Input(shape=(8,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(4, activation="softmax"),
])
rede.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
             loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])

historico = rede.fit(X_treino, y_treino, epochs=100, batch_size=16, verbose=0)

rede.summary()
print("acerto no treino:", round(historico.history["accuracy"][-1], 3))

## 6) Avaliar

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay)

probabilidades = rede.predict(X_teste, verbose=0)
y_pred = probabilidades.argmax(axis=1)

print("acuracia:", round(accuracy_score(y_teste, y_pred), 3))
print("f1_macro:", round(f1_score(y_teste, y_pred, average="macro"), 3))
print(classification_report(y_teste, y_pred, target_names=ORDEM, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_teste, y_pred, display_labels=ORDEM,
                                        xticks_rotation=45, cmap="Blues")
plt.tight_layout(); plt.show()

## 7) Converter para TensorFlow Lite

O conversor transforma a rede num arquivo `.tflite`: as camadas, os 212 pesos e a ordem das
contas, num formato que o interpretador da placa sabe ler. A conferência roda o `.tflite`
nas mesmas janelas de teste: as classes têm que ser as mesmas da rede do Keras.

In [ ]:
conversor = tf.lite.TFLiteConverter.from_keras_model(rede)
modelo_tflite = conversor.convert()

tflite_filename = "modelo_motor.tflite"
with open(tflite_filename, "wb") as f:
    f.write(modelo_tflite)
print(f"{tflite_filename}: {len(modelo_tflite)} bytes")

# Confere: o .tflite decide igual a rede do Keras?
interpretador = tf.lite.Interpreter(model_content=modelo_tflite)
interpretador.allocate_tensors()
entrada = interpretador.get_input_details()[0]["index"]
saida   = interpretador.get_output_details()[0]["index"]

y_tflite = []
for janela in X_teste:
    interpretador.set_tensor(entrada, janela.reshape(1, 8))
    interpretador.invoke()
    y_tflite.append(interpretador.get_tensor(saida)[0].argmax())

iguais = (np.array(y_tflite) == y_pred).sum()
print(f"o .tflite e a rede do Keras concordam em {iguais} de {len(y_pred)} janelas")

## 8) E se quantizar para int8?

Quantizar troca cada peso de 32 bits (float) por 8 bits (inteiro): o modelo fica até 4 vezes
menor, com um pouco menos de precisão. Vale para redes grandes, de imagem e de áudio. Aqui,
com 212 pesos, o arquivo é quase todo "envelope" (a descrição das camadas), e não pesos.
Rode e compare os tamanhos.

In [ ]:
conversor_int8 = tf.lite.TFLiteConverter.from_keras_model(rede)
conversor_int8.optimizations = [tf.lite.Optimize.DEFAULT]
conversor_int8.representative_dataset = lambda: ([x.reshape(1, 8)] for x in X_treino)
conversor_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conversor_int8.inference_input_type  = tf.int8
conversor_int8.inference_output_type = tf.int8
modelo_int8 = conversor_int8.convert()

print(f"float32: {len(modelo_tflite)} bytes | int8: {len(modelo_int8)} bytes")
print("Para a placa vai o float32.")

## 9) Exportar o scaler para C++

O mesmo arquivo do notebook da floresta: a média e o desvio de cada feature, e a função que
faz a conta na placa.

In [ ]:
scaler_filename = "ModeloMotorScaler.hpp"

means  = scaler.mean_.astype(float)
scales = scaler.scale_.astype(float)

conteudo = f"""#ifndef STANDARD_SCALER_HPP
#define STANDARD_SCALER_HPP

// Ordem: mean_ax, mean_ay, mean_az, std_ax, std_ay, std_az, std_mag, p2p_mag.
namespace Scaler {{
    const static float means[{len(means)}] = {{
        {", ".join(f"{m:.10f}f" for m in means)}
    }};

    const static float scales[{len(scales)}] = {{
        {", ".join(f"{s:.10f}f" for s in scales)}
    }};

    inline void standardize(const float* input, float* output) {{
        for (int i = 0; i < {len(means)}; i++) {{
            output[i] = (input[i] - means[i]) / scales[i];
        }}
    }}
}}

#endif
"""

with open(scaler_filename, "w") as f:
    f.write(conteudo)

print(conteudo)
print(f"Arquivo {scaler_filename} gerado.")

## 10) A rede vira um `.hpp`

A placa não tem sistema de arquivos para abrir um `.tflite`. Os bytes do arquivo viram um
vetor em C++ (`modeloMotorRede`), que é compilado junto com o firmware e fica na flash. No
`setup()`, o `ModelInit()` aponta o interpretador para esse vetor.

In [ ]:
rede_filename = "ModeloMotorRede.hpp"

linhas = []
for i in range(0, len(modelo_tflite), 12):
    linhas.append("  " + ", ".join(f"0x{b:02x}" for b in modelo_tflite[i:i + 12]) + ",")

conteudo = (
    "#ifndef MODELO_MOTOR_REDE_HPP\n#define MODELO_MOTOR_REDE_HPP\n\n"
    "// A rede neural do motor (8 -> 16 -> 4), no formato TFLite.\n"
    "// Gerado pelo notebook: nao edite a mao.\n"
    "alignas(16) const unsigned char modeloMotorRede[] = {\n"
    + "\n".join(linhas) +
    f"\n}};\nconst unsigned int modeloMotorRede_len = {len(modelo_tflite)};\n\n#endif\n"
)

with open(rede_filename, "w") as f:
    f.write(conteudo)

print("\n".join(conteudo.splitlines()[:8]))
print("...")
print(f"\nArquivo {rede_filename} gerado: {len(modelo_tflite)} bytes da rede, {conteudo.count(chr(10))} linhas.")
print("\nCole esta linha no .cpp do firmware:")
print('const char* NOMES_CLASSES[4] = { ' + ", ".join(f'"{c}"' for c in ORDEM) + ' };')

## 11) Baixar

Os dois `.hpp` vão direto para a pasta `src/` do firmware da torre, por cima dos que vieram
no projeto (os do dataset de referência). O `.tflite` é seu: é o mesmo modelo, para abrir em
qualquer ferramenta do TensorFlow Lite.

In [ ]:
import zipfile

# Um zip so: os tres arquivos saem do mesmo treino e viajam juntos.
with zipfile.ZipFile("modelo_motor_rede.zip", "w") as pacote:
    pacote.write(tflite_filename)
    pacote.write(scaler_filename)
    pacote.write(rede_filename)

try:
    from google.colab import files
    files.download("modelo_motor_rede.zip")
except Exception:
    print("modelo_motor_rede.zip gerado na pasta atual.")